In [1]:
import keras
from keras import layers,models
import keras_hub
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split 
import numpy as np
import os

C:\Users\danci\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):
C:\Users\danci\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_dir = "data/UTKFace"

def create_df(directory):
    files = [f for f in os.listdir(directory) if f.endswith('.jpg')]
    data = [] 
    
    for f in files:
        parts = f.split('_')
        if len(parts) >= 3:
            try:
                data.append({
                    'path': os.path.join(directory, f), 
                    'age': float(parts[0]),
                    'gender': int(parts[1]),
                    'race': int(parts[2])
                })
            except (ValueError, IndexError):
                continue
                
    return pd.DataFrame(data)

df = create_df(data_dir)

train_df, val_df = train_test_split(df, test_size=0.15, random_state=42)

print(df.head())



                                                path    age  gender  race
0  data/UTKFace\100_0_0_20170112213500903.jpg.chi...  100.0       0     0
1  data/UTKFace\100_0_0_20170112215240346.jpg.chi...  100.0       0     0
2  data/UTKFace\100_1_0_20170110183726390.jpg.chi...  100.0       1     0
3  data/UTKFace\100_1_0_20170112213001988.jpg.chi...  100.0       1     0
4  data/UTKFace\100_1_0_20170112213303693.jpg.chi...  100.0       1     0


In [7]:
# Replace your current UTKDataLoader class with this version
class UTKDataLoader(keras.utils.PyDataset):
    def __init__(self, df, batch_size=32, img_size=(224, 224), **kwargs):
        super().__init__(**kwargs)
        self.df = df
        self.batch_size = batch_size
        self.img_size = img_size

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def on_epoch_end(self):
        self.df = self.df.sample(frac=1).reset_index(drop=True)

    def __getitem__(self, idx):
        batch_df = self.df.iloc[idx * self.batch_size : (idx + 1) * self.batch_size]
        
        images = np.array([
            np.array(Image.open(p).convert('RGB').resize(self.img_size)) 
            for p in batch_df['path']
        ], dtype="float32")
        
        # --- Return a LIST of targets in specific order [Age, Gender, Race] ---
        age = (batch_df['age'].values / 100.0).astype("float32")
        gender = batch_df['gender'].values.astype("float32")
        race = batch_df['race'].values.astype("int32") 
        
        return images, (age, gender, race)


train_ds = UTKDataLoader(train_df)
val_ds = UTKDataLoader(val_df)

In [4]:
inputs = keras.Input(shape=(224,224,3))
x = layers.Rescaling(1./225)(inputs)

base_model = keras_hub.models.ResNetBackbone.from_preset("resnet_50_imagenet")
x = base_model(x)
x = layers.GlobalAveragePooling2D()(x)
age_h = models.Sequential([
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(1, name="age_output")
    ])(x)
    
gender_h = models.Sequential([
    layers.Dense(64, activation="relu"),
    layers.Dense(1, activation="sigmoid", name="gender_output")
])(x)
    
race_h = models.Sequential([
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(5, activation="softmax", name="race_output")
])(x)

model = keras.Model(inputs=inputs,outputs=[age_h,gender_h,race_h])
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ rescaling (Rescaling)         │ (None, 224, 224, 3)       │               0 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ res_net_backbone              │ (None, 7, 7, 2048)        │      23,561,152 │ rescaling[0][0]            │
│ (ResNetBackbone)              │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ global_average_pooling2d      │ (None, 2048)              │               0 │ res_net_backbone[0][0]     │
│ (GlobalAveragePooling2D)      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ sequential (Sequential)       │ (None, 1)                 │         262,401 │ global_average_pooling2d[… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ sequential_1 (Sequential)     │ (None, 1)                 │         131,201 │ global_average_pooling2d[… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ sequential_2 (Sequential)     │ (None, 5)                 │         262,917 │ global_average_pooling2d[… │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 24,217,671 (92.38 MB)

 Trainable params: 24,164,551 (92.18 MB)

 Non-trainable params: 53,120 (207.50 KB)

In [9]:
# Re-run your compile cell with this list format
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss=[
        "mse",                          # Age
        "binary_crossentropy",          # Gender
        "sparse_categorical_crossentropy" # Race
    ],
    loss_weights=[2.0, 1.0, 1.0],
    metrics=["mae", "accuracy","accuracy"]
)

In [10]:
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',      # Watch the validation loss (not training loss!)
    patience=3,              # Wait for 3 epochs of no improvement before quitting
    min_delta=0.001,         # Improvement must be at least this much to "count"
    restore_best_weights=True, # VERY IMPORTANT: Undo the overfitting by going back to the best weights
    verbose=1
)

checkpoint = keras.callbacks.ModelCheckpoint(
    "best_utk_model.keras",
    monitor="val_loss",
    save_best_only=True      # Only overwrites the file if the new model is better
)
# Train the model
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stop]
)

# Save the model
model.save("face_multi_task.keras")

Epoch 1/10
  1/630 ━━━━━━━━━━━━━━━━━━━━ 12:21:40 71s/step - loss: 2.7099 - sequential_1_accuracy: 0.5000 - sequential_1_loss: 0.7042 - sequential_2_accuracy: 0.2812 - sequential_2_loss: 1.5760 - sequential_loss: 0.2149 - sequential_mae: 0.4142

KeyboardInterrupt: 